# Hook으로 AgentCore Memory를 사용하는 Strands Agent 튜토리얼

## 개요

이 튜토리얼에서는 hook을 통해 AgentCore Memory와 통합된 Strands agent를 사용하여 지능형 개인 도우미를 구축하는 방법을 살펴봅니다. 에이전트는 대화 맥락을 유지하고 상호 작용에서 학습하여 개인화된 응답을 제공합니다.

## 튜토리얼 세부 정보

**사용 사례**: 수학 도우미

| 항목                | 세부 정보                                                                        |
|:--------------------|:---------------------------------------------------------------------------------|
| 튜토리얼 유형       | 장기 대화형                                                                       |
| Agent 유형          | 수학 도우미                                                                       |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                      |
| 튜토리얼 구성 요소  | 메모리용 AgentCore Summary Strategy, 메모리 저장 및 검색용 Hook                   |
| 예제 난이도         | 중급                                                                              |


다음 내용을 학습합니다.
- 대화 요약을 사용하도록 AgentCore Memory 설정
- 자동 저장 및 검색을 위한 메모리 hook 생성
- 지속형 메모리를 사용하는 Strands agent 구축
- 여러 대화에 걸친 메모리 기능 테스트
- 대체 학습 경로를 위한 대화 분기 사용
- 학생의 학습 진도와 성과를 추적하는 metadata 적용

### 시나리오 배경

이 예제에서는 이전 대화의 요약을 저장하는 수학 도우미를 만듭니다.
이 예제의 주요 기능은 다음과 같습니다.
- **자동 메모리 저장**: 대화를 자동으로 저장
- **맥락 검색**: 이전 대화를 현재 응답에 활용
- **요약 생성**: 핵심 정보를 추출하고 요약
- **Tool 통합**: 수학 연산을 위한 Calculator tool
- **대화 분기**: 다른 난이도와 교육 방식 탐색
- **Metadata 추적**: 난이도, 성과, 학습 milestone으로 event에 tag 지정

## 아키텍처
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>


## 사전 요구 사항

이 튜토리얼을 실행하려면 다음이 필요합니다.
- Python 3.10+
- Amazon Bedrock AgentCore Memory 권한이 있는 AWS credentials
- Amazon Bedrock AgentCore SDK

## 1단계: 환경 설정
이 Notebook 실행에 필요한 모든 library를 import하고 client를 정의하겠습니다.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
from bedrock_agentcore_starter_toolkit.operations.memory.manager import (
    MemoryManager,
)
from bedrock_agentcore_starter_toolkit.operations.memory.models.strategies import (
    SemanticStrategy,
)
from bedrock_agentcore.memory import MemorySessionManager
from bedrock_agentcore.memory.constants import (
    ConversationalMessage,
    MessageRole,
    RetrievalConfig,
)
from bedrock_agentcore.memory.models import (
    StringValue,
)

In [ ]:
import os
import logging
from strands import Agent
from datetime import datetime
from strands_tools import calculator
from strands.hooks import (
    AfterInvocationEvent,
    HookProvider,
    HookRegistry,
    MessageAddedEvent,
)

# Logging 설정
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("memory-tutorial")

# 구성 - 사용자 환경의 값으로 교체
REGION = os.getenv("AWS_REGION", "us-west-2")
ACTOR_ID = f"student-{datetime.now().strftime('%Y%m%d%H%M%S')}"
SESSION_ID = f"math-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# 코드를 간결하게 유지하도록 message role 상수 정의
USER = MessageRole.USER
ASSISTANT = MessageRole.ASSISTANT

## 2단계: Memory Resource 생성

이 단계에서는 semantic strategy를 사용하는 memory resource를 생성합니다. 이 resource는 대화 데이터를 저장하고 구성합니다. 기본 제공 SemanticStrategy는 IAM execution role 없이 대화에서 fact를 자동으로 포착합니다.


In [ ]:
# Memory Manager 초기화
memory_manager = MemoryManager(region_name=REGION)
memory_name = "MathAssistant"

# SemanticStrategy를 사용하여 memory strategy 정의
strategies = [
    SemanticStrategy(
        name="MathLearningMemory",
        description="Captures facts from math learning conversations",
        namespaces=["/students/math/{actorId}/"],
    )
]

# MemoryManager를 사용하여 memory resource 생성
memory_id = None  # Exception handler에서 NameError가 발생하지 않도록 초기화
try:
    memory = memory_manager.get_or_create_memory(
        name=memory_name,
        strategies=strategies,
        description="Memory for tutorial agent",
        event_expiry_days=30,
    )
    memory_id = memory.id
    logger.info(f"✅ Created memory: {memory_id}")
except Exception as e:
    # Memory 생성 중 발생하는 오류 처리
    logger.error(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()
    # 오류 발생 시 정리 - memory가 일부 생성되었다면 삭제
    if memory_id:
        try:
            memory_manager.delete_memory(memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.error(f"Failed to clean up memory: {cleanup_error}")
    # 실행을 중단하도록 exception을 다시 발생시킴
    raise

# Memory_id가 정상적으로 생성되었는지 확인
if memory_id is None:
    raise RuntimeError("Failed to create or retrieve memory ID")

## 3단계: Session Manager 초기화

이제 학생용 MemorySessionManager와 MemorySession을 생성합니다. Session manager는 모든 작업에서 memory_id, actor_id, session_id parameter를 자동으로 처리하여 더 간결한 API를 제공합니다.

이 session 기반 접근 방식은 메모리 작업을 단순화하고 코드 유지보수성을 높입니다.

In [ ]:
# Session manager 초기화
session_manager = MemorySessionManager(memory_id=memory_id, region_name=REGION)

# 특정 학생용 memory session 생성
student_session = session_manager.create_memory_session(actor_id=ACTOR_ID, session_id=SESSION_ID)

logger.info(f"✅ Session manager initialized for memory: {memory_id}")
logger.info(f"✅ Student session created for actor: {ACTOR_ID}")
logger.info(f"   Session ID: {SESSION_ID}")

## 4단계: Session을 지원하는 Memory Hook Provider 생성

이 단계에서는 MemorySession으로 메모리 작업을 자동화하는 사용자 지정 `MemoryHookProvider` class를 정의합니다. Hook은 에이전트 실행 lifecycle의 특정 시점에 실행되는 특수 function입니다. 여기서 생성하는 memory hook은 두 가지 주요 기능을 수행합니다.

1. **메모리 검색**: 사용자가 message를 보내면 `search_long_term_memories()`를 사용하여 관련 이전 대화를 자동으로 검색
2. **메모리 저장**: 에이전트가 응답한 후 ConversationalMessage object와 `add_turns()`를 사용하여 새 대화 저장

이를 통해 수동 관리 없이 원활한 메모리 환경을 구현할 수 있으며, session 기반 API를 사용하므로 memory_id, actor_id, session_id를 반복해서 전달할 필요가 없습니다.

In [ ]:
class MemoryHookProvider(HookProvider):
    """MemorySession으로 메모리를 자동 관리하는 훅 제공자입니다."""

    def __init__(self, student_session):
        """MemorySession 인스턴스로 초기화합니다.

        인자:
            student_session: 학생용 MemorySession 인스턴스
        """
        self.student_session = student_session

        # 수학 학습 맥락을 위한 검색 구성 정의
        self.retrieval_config = RetrievalConfig(top_k=5, relevance_score=0.3)

    def retrieve_memories(self, event: MessageAddedEvent):
        """사용자 메시지를 처리하기 전에 MemorySession으로 관련 메모리를 검색합니다."""
        messages = event.agent.messages
        if messages[-1]["role"] == "user" and "toolResult" not in messages[-1]["content"][0]:
            user_message = messages[-1]["content"][0].get("text", "")

            try:
                # 맥락 검색에 MemorySession 사용 (actor_id 전달 불필요)
                namespace_prefix = f"/students/math/{self.student_session._actor_id}/"

                # Session API를 사용하여 장기 메모리 검색
                memories = self.student_session.search_long_term_memories(
                    query=user_message,
                    namespace_prefix=namespace_prefix,
                    top_k=self.retrieval_config.top_k,
                )

                # 관련성 score로 필터링
                filtered_memories = [
                    memory for memory in memories if memory.get("score", 0) >= self.retrieval_config.relevance_score
                ]

                # Memory content 추출
                memory_context = []
                for memory in filtered_memories:
                    if isinstance(memory, dict):
                        content = memory.get("content", {})
                        if isinstance(content, dict):
                            text = content.get("text", "").strip()
                            score = memory.get("score", 0)
                            if text:
                                memory_context.append(f"[Score: {score:.2f}] {text}")

                # User message에 메모리 주입
                if memory_context:
                    context_text = "\n".join(memory_context)
                    original_text = messages[-1]["content"][0].get("text", "")
                    messages[-1]["content"][0]["text"] = f"{original_text}\n\nStudent Learning Context:\n{context_text}"
                    logger.info(
                        f"✅ Retrieved {len(memory_context)} relevant memories (filtered from {len(memories)} total)"
                    )

            except Exception as e:
                logger.error(f"Failed to retrieve memories: {e}")

    def save_memories(self, event: AfterInvocationEvent):
        """에이전트 응답 후 MemorySession으로 대화를 저장합니다."""
        try:
            messages = event.agent.messages
            if len(messages) >= 2 and messages[-1]["role"] == "assistant":
                # 마지막 user 및 assistant message 가져오기
                user_msg = None
                assistant_msg = None

                for msg in reversed(messages):
                    if msg["role"] == "assistant" and not assistant_msg:
                        assistant_msg = msg["content"][0]["text"]
                    elif msg["role"] == "user" and not user_msg and "toolResult" not in msg["content"][0]:
                        user_msg = msg["content"][0]["text"]
                        break

                if user_msg and assistant_msg:
                    # ConversationalMessage object와 함께 MemorySession 사용
                    interaction_messages = [
                        ConversationalMessage(user_msg, USER),
                        ConversationalMessage(assistant_msg, ASSISTANT),
                    ]

                    result = self.student_session.add_turns(interaction_messages)
                    logger.info(f"✅ Saved conversation using MemorySession - Event ID: {result['eventId']}")

        except Exception as e:
            logger.error(f"Failed to save memories: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        """메모리 훅을 등록합니다."""
        registry.add_callback(MessageAddedEvent, self.retrieve_memories)
        registry.add_callback(AfterInvocationEvent, self.save_memories)
        logger.info("✅ Memory hooks registered with MemorySession support")

## 5단계: Memory를 사용하는 Agent 생성

이제 Strands agent를 생성하고 MemorySession을 사용하는 memory hook provider와 연결합니다. 이 에이전트에는 두 가지 핵심 기능이 있습니다.

1. **Memory 통합**: 앞에서 생성한 memory hook이 session 기반 작업으로 맥락을 자동 검색
2. **Calculator Tool**: 필요할 때 에이전트가 수학 연산 수행

이 두 기능을 결합하면 학생의 진도를 기억하면서 필요한 계산도 수행하는 수학 튜터를 만들 수 있습니다.

In [ ]:
# MemorySession을 사용하는 memory hook provider 생성
memory_hooks = MemoryHookProvider(student_session)

# Memory hook 및 calculator tool을 사용하는 에이전트 생성
agent = Agent(
    hooks=[memory_hooks],
    model="global.anthropic.claude-haiku-4-5-20251001-v1:0",
    tools=[calculator],
    system_prompt="You are a helpful personal math tutor. You assist users in solving math problems and provide personalized assistance based on their learning progress and preferences.",
)

logger.info("✅ Agent created with MemorySession-based hooks")
logger.info(f"   Student: {ACTOR_ID}")
logger.info(f"   Session: {SESSION_ID}")

**에이전트 설정이 완료되었습니다. 이제 테스트해 보겠습니다.**

## Memory 기능 테스트

이 섹션에서는 일련의 상호 작용을 통해 에이전트의 메모리 기능을 테스트합니다. 에이전트가 시간이 지나면서 맥락을 구축하고 이전 상호 작용을 기억하는 방식을 살펴봅니다.

먼저 에이전트에게 자신을 소개하고 수학 문제를 질문해 보겠습니다.

In [ ]:
# 첫 번째 상호 작용 - 자기소개
response1 = agent(
    "Hi, I'm John and I just enrolled in Discrete Math course. Help me solve this: How many ways can I arrange 5 books on a shelf?"
)
print(f"Agent: {response1}")

에이전트에게 다른 계산 과제를 주겠습니다.

In [ ]:
# 두 번째 상호 작용 - 다른 계산
response2 = agent(
    "I learn better with step-by-step explanation with example questions. Can you explain modular arithmetic? What's 17 mod 5?"
)
print(f"Agent: {response2}")

이제 에이전트가 사용자를 기억하는지 확인해 보겠습니다.

**참고:** 메모리를 추출, 통합, 저장할 시간을 확보하도록 여기서 약 20초간 기다리세요.

In [ ]:
# 세 번째 상호 작용 - 메모리 회상 테스트
response3 = agent("I got that right! What's the immediate next step that I should study after modular arithmetic?")
print(f"Agent: {response3}")

마지막으로 에이전트가 계산 기록을 기억하는지 확인해 보겠습니다.

In [ ]:
# 네 번째 상호 작용 - 맥락 인식 테스트
response4 = agent("This is too hard, can we try something easier?")
print(f"Agent: {response4}")

### Memory 저장 확인

마지막 단계로 대화가 AgentCore Memory에 올바르게 저장되었는지 확인합니다. 이를 통해 memory hook이 제대로 작동하고 에이전트가 향후 상호 작용에서 이 정보에 액세스할 수 있음을 확인할 수 있습니다.

In [ ]:
# MemorySession을 사용하여 저장된 메모리 확인
try:
    namespace_prefix = f"/students/math/{ACTOR_ID}/"

    memories = student_session.search_long_term_memories(
        query="mathematics calculations learning progress",
        namespace_prefix=namespace_prefix,
        top_k=5,
    )

    print(f"\n📚 Found {len(memories)} memories for student {ACTOR_ID}:")
    print("=" * 60)
    for i, memory in enumerate(memories, 1):
        if isinstance(memory, dict):
            content = memory.get("content", {})
            score = memory.get("score", 0)
            if isinstance(content, dict):
                text = content.get("text", "")[:200] + "..."
                print(f"\n{i}. [Relevance: {score:.2f}]")
                print(f"   {text}")
    print("\n" + "=" * 60)

except Exception as e:
    logger.error(f"Error retrieving memories: {e}")

## 고급 기능: Branching 및 Metadata

### 대화 Branching

Branching을 사용하면 어느 지점에서든 다른 대화 경로를 탐색할 수 있습니다. 다음 작업에 유용합니다.
- 서로 다른 난이도 테스트
- 대체 설명 방식 탐색
- 교육 방식 A/B 테스트

더 어려운 문제 경로를 탐색하는 branch를 생성해 보겠습니다.

In [ ]:
# 대화의 마지막 event ID 가져오기
events = student_session.list_events()
if events:
    last_event_id = events[-1].eventId

    # 고급 주제를 탐색하도록 대화 fork
    branch_event = student_session.fork_conversation(
        root_event_id=last_event_id,
        branch_name="advanced-path",
        messages=[
            ConversationalMessage(
                "Actually, I'm ready for a challenge! Can you give me a harder problem involving modular arithmetic and combinatorics?",
                USER,
            ),
            ConversationalMessage(
                "Great! Here's a challenging problem: How many 4-digit numbers are there where the sum of digits is congruent to 3 (mod 5)? This combines modular arithmetic with counting principles.",
                ASSISTANT,
            ),
        ],
    )

    logger.info(f"✅ Created branch 'advanced-path' from event {last_event_id}")
    logger.info(f"   Branch event ID: {branch_event['eventId']}")

    # 모든 branch 나열
    branches = student_session.list_branches()
    print(f"\n🌳 Session has {len(branches)} branch(es):")
    for branch in branches:
        print(f"   - {branch.name}: {branch.event_count} events")

    # 고급 branch의 event 가져오기
    advanced_events = student_session.list_events(branch_name="advanced-path")
    print(f"\n📋 Advanced branch has {len(advanced_events)} events")
else:
    print("No events found to branch from")

### Metadata 활용

Metadata를 사용하면 event에 사용자 지정 정보를 tag로 지정하여 더 효율적으로 구성하고 검색할 수 있습니다. Metadata로 다음 항목을 추적해 보겠습니다.
- 문제 난이도
- 학생 성과
- 주제 범주
- 학습 milestone

In [ ]:
# 풍부한 metadata가 포함된 새 상호 작용 추가
metadata_event = student_session.add_turns(
    messages=[
        ConversationalMessage(
            "Let me try: If I choose 3 books from 5, that's C(5,3) = 10 ways, right?",
            USER,
        ),
        ConversationalMessage(
            "Excellent! You correctly applied the combination formula. That's exactly right: C(5,3) = 5!/(3!×2!) = 10.",
            ASSISTANT,
        ),
    ],
    metadata={
        "difficulty": StringValue.build("intermediate"),
        "topic": StringValue.build("combinatorics"),
        "subtopic": StringValue.build("combinations"),
        "performance": StringValue.build("correct"),
        "milestone": StringValue.build("first_correct_combination"),
        "learning_stage": StringValue.build("applying_formulas"),
    },
)

logger.info(f"✅ Added event with metadata - Event ID: {metadata_event['eventId']}")
print("\n📊 Event tagged with:")
print("   - Difficulty: intermediate")
print("   - Topic: combinatorics")
print("   - Performance: correct")
print("   - Milestone: first_correct_combination")

### Metadata로 Event 질의

이제 metadata를 기준으로 event를 필터링하여 학생의 진도를 분석할 수 있습니다.

In [ ]:
# 학생이 정답을 맞힌 event 질의
try:
    correct_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "performance"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "correct"}},
            }
        ]
    )

    print(f"\n✅ Found {len(correct_events)} event(s) where student answered correctly")

    # 중급 난이도 문제 질의
    intermediate_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "difficulty"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "intermediate"}},
            }
        ]
    )

    print(f"📈 Found {len(intermediate_events)} intermediate difficulty problem(s)")

    # 조합론 주제 질의
    combinatorics_events = student_session.list_events(
        eventMetadata=[
            {
                "left": {"metadataKey": "topic"},
                "operator": "EQUALS_TO",
                "right": {"metadataValue": {"stringValue": "combinatorics"}},
            }
        ]
    )

    print(f"🎯 Found {len(combinatorics_events)} combinatorics-related event(s)")

    print("\n💡 Use cases for metadata:")
    print("   - Track student progress by difficulty level")
    print("   - Identify topics needing more practice")
    print("   - Generate performance reports")
    print("   - Personalize learning paths based on history")

except Exception as e:
    logger.error(f"Error querying metadata: {e}")
    print("Note: Metadata filtering requires events with metadata tags")

튜토리얼을 완료했습니다! 🎉

핵심 내용:
- Memory hook은 대화 맥락을 자동으로 저장하고 검색합니다.
- 에이전트는 여러 상호 작용에 걸쳐 state를 유지할 수 있습니다.
- AgentCore Memory는 관련 맥락을 찾는 semantic search를 제공합니다.
- Tool과 메모리를 결합하여 기능을 강화할 수 있습니다.
- **Branching을 통해 다른 대화 경로를 탐색할 수 있습니다.**
- **Metadata는 강력한 필터링 및 분석 기능을 제공합니다.**

## 정리

### 선택 사항: Memory Resource 삭제

튜토리얼을 완료한 후 불필요한 비용이 발생하지 않도록 memory resource를 삭제할 수 있습니다. 다음 정리 코드는 기본적으로 주석 처리되어 있습니다.

In [ ]:
# Memory resource를 삭제하려면 주석 해제
# try:
#     memory_manager.delete_memory(memory_id)
#     print(f"✅ Deleted memory resource: {memory_id}")
# except Exception as e:
#     print(f"Error deleting memory: {e}")